In [1]:
import os, sys, json, warnings, datetime, traceback, pathlib, io as _io
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.join(os.getcwd(), "qens_library"))

import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, LinearSegmentedColormap
from scipy.optimize import curve_fit, nnls
from scipy.signal import fftconvolve
from scipy.ndimage import gaussian_filter
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

import qens
from qens.config        import Config
from qens.constants     import hbar_mevps
from qens.io            import load_dataset, read_nxspe_hdf5, inspect_nxspe
from qens.preprocessing import fit_elastic_peak, assign_resolution
from qens.models        import ce, fickian, ss_model, lorentz, gnorm
from qens.fitting       import extract_hwhm, save_hwhm_csv, build_data_bins, find_map
from qens.sampling      import run_mcmc

try:
    import emcee
    _EMCEE = True
except ImportError:
    _EMCEE = False

_MODEL_COLORS = {
    "ce":       "#c0392b",
    "fickian":  "#2471a3",
    "ss_model": "#1e8449",
    "lorentz":  "#6c3483",
}

_MODEL_LABELS = {
    "ce":       "Chudley-Elliott",
    "fickian":  "Fickian",
    "ss_model": "Singwi-Sjolander",
    "lorentz":  "Lorentzian",
}

_CMAP = LinearSegmentedColormap.from_list(
    "qens", ["#0a0e1a","#0c2d6b","#1565c0","#42a5f5","#e3f2fd","#ff8f00","#e65100"], N=512)

plt.rcParams.update({
    "font.family":     "DejaVu Sans",
    "axes.linewidth":  0.8,
    "axes.edgecolor":  "#555",
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "legend.framealpha": 0.92,
    "legend.edgecolor":  "#ccc",
    "figure.dpi":        100,
})

print(f"qens_library  {qens.__version__}")
print(f"emcee         {_EMCEE}")
print("Initialised.")

qens_library  0.2.0
emcee         True
Initialised.


In [2]:
# Model fitting and figure functions


_custom_models = {}
_MODEL_COLORS_CUSTOM = {}
_MODEL_LABELS_CUSTOM = {}


def _fit_gamma(q_arr, g_arr, model_name):
    try:
        if model_name == "ce":
            p, _ = curve_fit(ce, q_arr, g_arr, p0=[0.30, 2.5], bounds=([0,0.5], [3,6]))
            return {"D": p[0], "l": p[1], "tau": p[1]**2/(6*p[0])}
        elif model_name == "fickian":
            p, _ = curve_fit(fickian, q_arr, g_arr, p0=[0.30], bounds=([0], [3]))
            return {"D": p[0]}
        elif model_name == "ss_model":
            p, _ = curve_fit(ss_model, q_arr, g_arr, p0=[0.30,1.0], bounds=([0,0.01], [3,20]))
            return {"D": p[0], "tau_s": p[1]}
        elif model_name in _custom_models:
            info   = _custom_models[model_name]
            func   = info["func"]
            p0_v   = [pd[1] for pd in info["param_defs"]]
            lo_v   = [pd[2] for pd in info["param_defs"]]
            hi_v   = [pd[3] for pd in info["param_defs"]]
            p, _   = curve_fit(func, q_arr, g_arr, p0=p0_v, bounds=(lo_v, hi_v), maxfev=8000)
            res    = {}
            for i, pd in enumerate(info["param_defs"]):
                res[pd[0]] = float(p[i])
            return res
        else:  # lorentz
            return {"mean_hwhm_mev": float(np.mean(g_arr))}
    except Exception as exc:
        return {"error": str(exc)}




def _despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)




def fig_sqw_map(d_inc, ewin=1.0):
    good = d_inc["good"]
    q = d_inc["q"]
    e = d_inc["e"]
    emask = (e >= -ewin) & (e <= ewin)
    qs = np.argsort(q[good])
    img = d_inc["data"][np.ix_(good, emask)]
    img = np.where(np.isfinite(img) & (img > 0), img, np.nan)
    ism = gaussian_filter(np.where(np.isfinite(img[qs]), img[qs], 0), sigma=[1.5,0.8])
    ism[ism <= 0] = np.nan
    vmin, vmax = np.nanpercentile(ism, 2), np.nanpercentile(ism, 99)

    fig, ax = plt.subplots(figsize=(6.5, 4.8))
    im = ax.pcolormesh(e[emask], q[good][qs], ism, cmap=_CMAP,
                       norm=LogNorm(vmin=max(vmin,1e-6), vmax=vmax),
                       shading="auto", rasterized=True)
    ax.axvline(0, color="#888", lw=0.8, ls="--")
    ax.set_xlabel(r"Energy transfer  $\hbar\omega$  (meV)", fontsize=11)
    ax.set_ylabel(r"Momentum transfer  $Q$  ($\AA^{-1}$)", fontsize=11)
    ax.set_title(r"$S(Q,\omega)$  measured intensity", fontsize=11, pad=8)
    cb = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.038)
    cb.set_label(r"$S(Q,\omega)$  (a.u.)", fontsize=9)
    cb.ax.tick_params(labelsize=8)
    ax.tick_params(labelsize=9)
    fig.tight_layout()
    return fig




def fig_fit_residuals(d_inc, d_map, l_map, q_target=1.06):
    gp = d_inc["good"]
    qg = d_inc["q"][gp]
    sr = d_inc["sigma_res"]
    emask = (d_inc["e"] >= -0.8) & (d_inc["e"] <= 0.8)
    ew = d_inc["e"][emask]
    near = np.where(np.abs(qg - q_target) < 0.10)[0]
    if len(near) == 0:
        near = np.argsort(np.abs(qg - q_target))[:4]
    spec = np.nanmean([d_inc["data"][gp[j]][emask] for j in near], axis=0)
    errs = np.sqrt(np.nanmean([d_inc["errs"][gp[j]][emask]**2 for j in near], axis=0))
    spec = np.where(np.isfinite(spec), spec, 0)
    errs = np.where(errs > 0, errs, spec.max() * 0.05)
    sn = spec / spec.max()
    en = errs / spec.max()

    wf = np.linspace(-0.8, 0.8, 1000)
    dt = wf[1] - wf[0]
    gamma = ce(q_target, d_map, l_map)
    el = gnorm(wf, sr); el /= el.max()
    ql_ = fftconvolve(lorentz(wf, gamma), gnorm(wf, sr), mode="same") * dt
    ql = ql_ / ql_.max() if ql_.max() > 0 else ql_
    amp, _ = nnls(np.column_stack([el, ql, np.ones(len(wf))]), np.interp(wf, ew, sn))
    fit = amp[0]*el + amp[1]*ql + amp[2]
    fit_d = np.interp(ew, wf, fit)
    resid = (sn - fit_d) / en
    chi2r = float(np.sum(resid**2) / max(len(resid) - 4, 1))

    fig, axes = plt.subplots(2,1, figsize=(6.8,6.0),
                             gridspec_kw={"height_ratios":[3,1]}, sharex=True)
    axes[0].errorbar(ew, sn, yerr=en, fmt=".", color="#333", ms=3.5,
                     elinewidth=0.7, alpha=0.8, label="Data", zorder=5)
    axes[0].fill_between(wf, amp[2], amp[0]*el + amp[2],
                         alpha=0.22, color="#1565c0", label="Elastic")
    axes[0].fill_between(wf, amp[2], amp[1]*ql + amp[2],
                         alpha=0.22, color="#e65100", label="Quasi-elastic")
    axes[0].plot(wf, fit, "-", color="#c0392b", lw=2.0,
                 label=rf"CE model  $\chi^2_r={chi2r:.2f}$")
    axes[0].set_ylabel(r"$S(Q,\omega)$  normalised", fontsize=11)
    axes[0].set_title(rf"Single-spectrum fit  $Q={q_target:.2f}$ $\AA^{{-1}}$",
                      fontsize=11, pad=8)
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.18, lw=0.6)
    _despine(axes[0])

    axes[1].axhline(0, color="#555", lw=0.9)
    axes[1].axhline( 2, color="#999", ls="--", lw=0.7)
    axes[1].axhline(-2, color="#999", ls="--", lw=0.7)
    axes[1].plot(ew, resid, ".", color="#c0392b", ms=3.5, alpha=0.85)
    axes[1].set_xlabel(r"Energy transfer  $\hbar\omega$  (meV)", fontsize=11)
    axes[1].set_ylabel(r"Residual ($\sigma$)", fontsize=10)
    axes[1].set_ylim(-4.8, 4.8)
    axes[1].grid(True, alpha=0.18, lw=0.6)
    _despine(axes[1])

    fig.tight_layout(h_pad=0.4)
    return fig, chi2r




def fig_hwhm_comparison(q_hwhm, g_hwhm, g_err, model_results,
                         samples_map=None, res_hwhm_uev=50):
    q2d = q_hwhm**2
    q_f = np.linspace(max(q_hwhm.min()*0.85, 0.05), q_hwhm.max()*1.10, 350)
    q2f = q_f**2

    fig, ax = plt.subplots(figsize=(7.2, 5.2))

    for model, ls_res in model_results.items():
        col = _MODEL_COLORS.get(model, "#333")
        lbl = _MODEL_LABELS.get(model, model)
        smp = (samples_map or {}).get(model)

        if smp is not None:
            d_s, l_s = smp[:,0], np.abs(smp[:,1])
            idx = np.random.default_rng(0).choice(len(d_s), min(300, len(d_s)), replace=False)
            fan = np.array([ce(q_f, d_s[i], l_s[i])*1000 for i in idx])
            ax.fill_between(q2f, np.percentile(fan,2.5,axis=0), np.percentile(fan,97.5,axis=0),
                            alpha=0.14, color=col)

        if "error" in ls_res:
            continue

        D = ls_res.get("D", 0.30)
        L = ls_res.get("l", 2.50)
        ts = ls_res.get("tau_s", 1.0)
        c2 = ls_res.get("_chi2r")
        tag = (rf"$D={D:.4f}$" + (rf", $\ell={L:.4f}$" if model=="ce" else
               rf", $\tau_s={ts:.3f}$" if model=="ss_model" else ""))
        suffix = rf"  ($\chi^2_r={c2:.3f}$)" if c2 is not None else ""

        if model == "ce":
            y = ce(q_f, D, L) * 1000
        elif model == "fickian":
            y = fickian(q_f, D) * 1000
        elif model == "ss_model":
            y = ss_model(q_f, D, ts) * 1000
        elif model in _custom_models:
            info  = _custom_models[model]
            func  = info["func"]
            pvals = [ls_res.get(pd[0], pd[1]) for pd in info["param_defs"]]
            y = func(q_f, *pvals) * 1000
            parts = [f"{pd[0]}={ls_res.get(pd[0], pd[1]):.4f}" for pd in info["param_defs"]]
            tag = "  ".join(parts)
            suffix = rf"  ($\chi^2_r={c2:.3f}$)" if c2 is not None else ""
        else:
            mh = ls_res.get("mean_hwhm_mev", g_hwhm.mean()) * 1000
            ax.axhline(mh, color=col, lw=1.8, ls="-.", label=rf"{lbl}  $\bar{{\Gamma}}={mh:.0f}$ $\mu$eV")
            continue

        ax.plot(q2f, y, "-", color=col, lw=2.0, zorder=5, label=f"{lbl}  {tag}{suffix}")

    ax.errorbar(q2d, g_hwhm*1000, yerr=2*g_err*1000, fmt="o", color="#111", ms=5.5,
                capsize=3.5, elinewidth=1.4, label=r"Data  $\pm 2\sigma$", zorder=6)
    ax.axhline(res_hwhm_uev, color="#888", ls=":", lw=1.3,
               label=rf"Resolution HWHM  $={res_hwhm_uev:.0f}$ $\mu$eV")
    ax.axhspan(0, res_hwhm_uev*1.15, alpha=0.04, color="#888")

    ax.set_xlabel(r"$Q^2$  ($\AA^{-2}$)", fontsize=11)
    ax.set_ylabel(r"$\Gamma(Q)$  ($\mu$eV)", fontsize=11)
    ax.set_title(r"$\Gamma(Q)$ vs $Q^2$" + (" — model comparison" if len(model_results)>1 else ""),
                 fontsize=11, pad=8)
    ax.legend(fontsize=8.5, loc="upper left")
    ax.set_xlim(left=-0.04)
    ax.set_ylim(bottom=-8)
    ax.grid(True, alpha=0.18, lw=0.6)
    _despine(ax)
    fig.tight_layout()
    return fig




def fig_posteriors_multi(samples_map, model_results, d_inc):
    entries = [(m, s) for m, s in samples_map.items()]
    if not entries:
        return None
    n = len(entries)
    fig, axes = plt.subplots(n, 3, figsize=(12, 3.6*n), squeeze=False)
    for row, (model, smp) in enumerate(entries):
        col = _MODEL_COLORS.get(model, "#333")
        lbl = _MODEL_LABELS.get(model, model)
        d_s = smp[:,0]
        l_s = np.abs(smp[:,1])
        tau_s = l_s**2 / (6*d_s)
        for ci, (arr, name, unit) in enumerate([
                (d_s,   r"$D$",           r"$\AA^2\,\mathrm{ps}^{-1}$"),
                (l_s,   r"$\ell$",        r"$\AA$"),
                (tau_s, r"$\tau$",        r"$\mathrm{ps}$")]):
            ax = axes[row][ci]
            med = np.median(arr)
            lo, hi = np.percentile(arr, [2.5, 97.5])
            cnt, _, _ = ax.hist(arr, bins=60, density=True, color=col, alpha=0.75,
                                edgecolor="white", lw=0.2)
            pk = cnt.max()
            ax.axvspan(lo, hi, alpha=0.16, color=col)
            ax.axvline(med, color="#111", lw=1.8, label=rf"Median$={med:.4f}$")
            ax.annotate("", xy=(hi, pk*1.08), xytext=(lo, pk*1.08),
                        arrowprops=dict(arrowstyle="<->", color=col, lw=1.6))
            ax.text((lo+hi)/2, pk*1.14, rf"95% CI  [{lo:.4f}, {hi:.4f}]",
                    ha="center", fontsize=7.5, color=col)
            ax.set_xlabel(f"{name}  ({unit})", fontsize=10)
            ax.set_ylabel("Density", fontsize=9)
            ax.set_ylim(0, pk*1.28)
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.15, lw=0.6)
            _despine(ax)
            if ci == 0:
                ax.set_title(lbl, fontsize=11, color=col, fontweight="bold", loc="left")
    fig.suptitle("Posterior distributions", fontsize=12, fontweight="bold", y=1.01)
    fig.tight_layout()
    return fig




def fig_data_preview(d):
    good = d["good"]
    q_arr = d["q"][good]
    e = d["e"]
    ei = d["ei"]

    ewin = min(ei*0.45, 1.2)
    emask = (e >= -ewin) & (e <= ewin)
    qs = np.argsort(q_arr)
    img = d["data"][np.ix_(good, emask)]
    img = np.where(np.isfinite(img) & (img>0), img, np.nan)
    ism = gaussian_filter(np.where(np.isfinite(img[qs]), img[qs], 0.0), sigma=[1.5,0.8])
    ism[ism <= 0] = np.nan
    vmin = max(np.nanpercentile(ism, 2), 1e-8)
    vmax = np.nanpercentile(ism, 99)

    n_lo = max(3, len(good)//3)
    avg = np.nanmean([d["data"][good[j]] for j in range(n_lo)], axis=0)
    avg = np.where(np.isfinite(avg), avg, 0.0)
    emask2 = (e >= -ewin*0.6) & (e <= ewin*0.6)

    emask_el = (e >= -0.15) & (e <= 0.15)
    peak_heights = np.array([d["data"][i][emask_el].max() if emask_el.any() else 0.0 for i in good])
    peak_heights = np.where(np.isfinite(peak_heights), peak_heights, 0.0)

    fig = plt.figure(figsize=(13, 5.0))
    gs = fig.add_gridspec(2,2, width_ratios=[1,1], height_ratios=[5,1], hspace=0.08, wspace=0.32)
    ax_map   = fig.add_subplot(gs[:,0])
    ax_peak  = fig.add_subplot(gs[0,1])
    ax_strip = fig.add_subplot(gs[1,1])

    im = ax_map.pcolormesh(e[emask], q_arr[qs], ism, cmap=_CMAP,
                           norm=LogNorm(vmin=vmin, vmax=vmax), shading="auto", rasterized=True)
    ax_map.axvline(0.0, color="#aaa", lw=0.8, ls="--", alpha=0.6)
    ax_map.set_xlabel(r"$\hbar\omega$  (meV)", fontsize=11)
    ax_map.set_ylabel(r"$Q$  (Å$^{-1}$)", fontsize=11)
    ax_map.set_title(rf"$S(Q,\omega)$  —  {d['name']}", fontsize=10, pad=7)
    cb = fig.colorbar(im, ax=ax_map, pad=0.02, fraction=0.038)
    cb.set_label("intensity", fontsize=8)
    cb.ax.tick_params(labelsize=7)
    ax_map.tick_params(labelsize=9)
    _despine(ax_map)

    norm_pk = avg[emask2].max() if avg[emask2].max()>0 else 1.0
    ax_peak.fill_between(e[emask2], 0, avg[emask2]/norm_pk, alpha=0.35, color="#2471a3")
    ax_peak.plot(e[emask2], avg[emask2]/norm_pk, color="#2471a3", lw=1.8)
    ax_peak.axvline(0.0, color="#aaa", lw=0.8, ls="--", alpha=0.6)
    ax_peak.set_ylabel("normalised intensity", fontsize=9)
    ax_peak.set_title(f"Elastic peak  (lowest-Q detectors)  FWHM ≈ {d.get('fwhm_res',0)*1000:.0f} µeV  [{d.get('res_source','—')}]",
                      fontsize=8.5, pad=6)
    ax_peak.tick_params(labelsize=8)
    ax_peak.set_xlim(e[emask2][0], e[emask2][-1])
    ax_peak.set_ylim(bottom=-0.04)
    ax_peak.grid(True, alpha=0.18, lw=0.6)
    _despine(ax_peak)
    plt.setp(ax_peak.get_xticklabels(), visible=False)

    strip = peak_heights[np.argsort(q_arr)][np.newaxis,:]
    ax_strip.imshow(strip, aspect="auto", cmap="plasma",
                    extent=[q_arr.min(), q_arr.max(), 0, 1],
                    vmin=0, vmax=max(peak_heights.max(), 1e-8))
    ax_strip.set_xlabel(r"$Q$  (Å$^{-1}$)", fontsize=9)
    ax_strip.set_yticks([])
    ax_strip.set_title("detector peak intensity  (dark = weak / masked)", fontsize=8, pad=4)
    ax_strip.tick_params(labelsize=8)

    fig.tight_layout(pad=1.2)
    return fig

print("Backend functions loaded.")

Backend functions loaded.


In [3]:
_CSS = """
<style>
.qs-app * { box-sizing: border-box; font-family: "Helvetica Neue", Arial, sans-serif; }
.qs-heading {
    font-size: .74em; font-weight: 700; letter-spacing: .10em;
    text-transform: uppercase; color: #4a5568;
    border-bottom: 1px solid #e2e8f0; padding-bottom: 5px; margin-bottom: 10px;
}
.qs-card {
    background: #ffffff; border: 1px solid #e2e8f0;
    border-radius: 6px; padding: 14px 16px; margin-bottom: 10px;
}
.qs-lbl {
    font-size: .80em; font-weight: 600; color: #4a5568;
    margin-bottom: 3px; display: block;
}
.qs-ok   { background:#f0fdf4; border:1px solid #86efac; border-radius:5px; padding:9px 14px; color:#15803d; font-size:.88em; font-weight:600; }
.qs-err  { background:#fef2f2; border:1px solid #fca5a5; border-radius:5px; padding:9px 14px; color:#b91c1c; font-size:.87em; }
.qs-info { background:#eff6ff; border:1px solid #bfdbfe; border-radius:5px; padding:8px 14px; color:#1d4ed8; font-size:.87em; }
.qs-warn { background:#fffbeb; border:1px solid #fcd34d; border-radius:5px; padding:8px 14px; color:#92400e; font-size:.87em; }
.qs-tbl { border-collapse:collapse; width:100%; font-size:.87em; }
.qs-tbl th { background:#1e3a5f; color:#fff; padding:6px 12px; text-align:left; font-weight:600; }
.qs-tbl td { padding:5px 12px; border-bottom:1px solid #e2e8f0; }
.qs-tbl tr:nth-child(even) td { background:#f8fafc; }
.qs-tbl .qs-section td { background:#334155; color:#cbd5e1; font-size:.80em; letter-spacing:.08em; text-transform:uppercase; padding:4px 12px; }
.qs-tbl .qs-model-ce       td { background:#fdf2f2; color:#7f1d1d; }
.qs-tbl .qs-model-fickian  td { background:#eff6ff; color:#1e3a8a; }
.qs-tbl .qs-model-ss_model td { background:#f0fdf4; color:#14532d; }
.qs-tbl .qs-model-lorentz  td { background:#faf5ff; color:#4c1d95; }
.qs-tbl .qs-model-custom   td { background:#fefce8; color:#713f12; }
.qs-log-ok   { color: #15803d; }
.qs-log-warn { color: #b45309; }
.qs-log-err  { color: #b91c1c; }
.qs-log-ts   { color: #94a3b8; font-size:.80em; margin-right:6px; }
.qs-path {
    font-family: "SFMono-Regular", "Consolas", monospace;
    font-size: .79em; color: #1e3a5f; background: #f1f5f9;
    border: 1px solid #cbd5e1; border-radius: 4px;
    padding: 4px 10px; margin-bottom: 6px; word-break: break-all;
}
.qs-badge {
    display: inline-block; background: #1e3a5f; color: #fff;
    font-size: .75em; padding: 1px 9px; border-radius: 12px;
    vertical-align: middle; margin-left: 6px;
}
.qs-cm-formula {
    font-family: "SFMono-Regular", "Consolas", monospace;
    font-size: .82em; background: #f8fafc;
    border: 1px solid #cbd5e1; border-radius: 4px;
    padding: 6px 8px;
}
.qs-cm-tag {
    display: inline-block; background: #78350f; color: #fef3c7;
    font-size: .73em; padding: 2px 8px; border-radius: 10px;
    margin: 2px 4px 2px 0; font-family: monospace;
}
</style>
"""




def register_custom_model(key, label, formula_str, param_defs, color="#78350f"):
    import re
    key = re.sub(r'[^a-z0-9_]', '_', key.lower().strip())
    if not key:
        raise ValueError("Model key cannot be empty.")
    ns = {"np": np, "hbar_mevps": hbar_mevps, "ce": ce, "fickian": fickian, "ss_model": ss_model}
    args = ", ".join(["q"] + [pd[0] for pd in param_defs])
    func = eval(f"lambda {args}: {formula_str}", ns)
    test_q = np.array([0.5, 1.0, 1.5])
    test_args = [test_q] + [pd[1] for pd in param_defs]
    result = func(*test_args)
    if not np.all(np.isfinite(result)):
        raise ValueError("Formula returned non-finite values.")
    _custom_models[key] = {
        "func": func, "param_defs": param_defs, "color": color,
        "label": label, "formula": formula_str
    }
    _MODEL_COLORS[key] = color
    _MODEL_LABELS[key] = label
    return True




def unregister_custom_model(key):
    _custom_models.pop(key, None)
    _MODEL_COLORS.pop(key, None)
    _MODEL_LABELS.pop(key, None)




print("CSS and custom model registry loaded.")

CSS and custom model registry loaded.


In [4]:
_w_cm_key = widgets.Text(value="", placeholder="e.g. my_model (no spaces)", layout=widgets.Layout(width="100%"))

_w_cm_label = widgets.Text(value="", placeholder="e.g. My Diffusion Model", layout=widgets.Layout(width="100%"))
_w_cm_formula = widgets.Textarea(
    value="", placeholder="Python expression for Gamma(Q) in meV.\nAvailable: q (array), np, hbar_mevps, ce, fickian, ss_model, and your parameter names.\nExamples:\n  hbar_mevps * D * q**2\n  hbar_mevps / tau * (1 - np.sinc(q*l/np.pi))",
    layout=widgets.Layout(width="100%", height="110px"))
_w_cm_color = widgets.Dropdown(
    options=[("Amber","#92400e"),("Teal","#0f766e"),("Indigo","#3730a3"),("Rose","#9f1239"),
             ("Cyan","#0e7490"),("Slate","#334155"),("Lime","#3f6212"),("Orange","#c2410c")],
    value="#92400e", layout=widgets.Layout(width="100%"))




_cm_param_rows = []
_cm_params_vbox = widgets.VBox([], layout=widgets.Layout(margin="4px 0"))
_cm_params_status = widgets.HTML("")




def _make_param_row():
    w_name = widgets.Text(placeholder="name (e.g. D)", layout=widgets.Layout(width="90px"))
    w_p0 = widgets.FloatText(value=0.30, layout=widgets.Layout(width="80px"))
    w_lo = widgets.FloatText(value=0.0, layout=widgets.Layout(width="80px"))
    w_hi = widgets.FloatText(value=10.0, layout=widgets.Layout(width="80px"))
    w_rm = widgets.Button(description="Remove", button_style="danger", layout=widgets.Layout(width="74px", height="28px"))
    row_dict = dict(name=w_name, p0=w_p0, lo=w_lo, hi=w_hi, rm=w_rm)
    row_widget = widgets.HBox([w_name, w_p0, w_lo, w_hi, w_rm],
                              layout=widgets.Layout(align_items="center", margin="2px 0"))
    def _remove(_):
        if row_dict in _cm_param_rows:
            _cm_param_rows.remove(row_dict)
            _refresh_params_vbox()
    w_rm.on_click(_remove)
    return row_widget, row_dict



def _refresh_params_vbox():
    if not _cm_param_rows:
        _cm_params_vbox.children = [widgets.HTML('<span style="color:#94a3b8;font-size:.82em">No parameters added yet.</span>')]
        return
    header = widgets.HBox([
        widgets.HTML('<span style="color:#64748b;font-size:.78em;width:90px;display:inline-block">Name</span>'),
        widgets.HTML('<span style="color:#64748b;font-size:.78em;width:80px;display:inline-block;margin-left:4px">p0</span>'),
        widgets.HTML('<span style="color:#64748b;font-size:.78em;width:80px;display:inline-block;margin-left:4px">Lower</span>'),
        widgets.HTML('<span style="color:#64748b;font-size:.78em;width:80px;display:inline-block;margin-left:4px">Upper</span>')
    ], layout=widgets.Layout(margin="0 0 2px 0"))
    rows = [header] + [rd["_widget"] for rd in _cm_param_rows]
    _cm_params_vbox.children = rows




def _add_param_row(_=None):
    row_widget, row_dict = _make_param_row()
    row_dict["_widget"] = row_widget
    _cm_param_rows.append(row_dict)
    _refresh_params_vbox()

_btn_add_param = widgets.Button(description="Add parameter", layout=widgets.Layout(width="130px", height="28px"))
_btn_add_param.on_click(_add_param_row)

_btn_register = widgets.Button(description="Register Model", button_style="primary",
                               layout=widgets.Layout(width="150px", height="34px"))
_w_register_status = widgets.HTML("")
_w_registered_list = widgets.VBox([widgets.HTML('<span style="color:#94a3b8;font-size:.82em">No custom models registered yet.</span>')])




def _rebuild_registered_list():
    if not _custom_models:
        _w_registered_list.children = [widgets.HTML('<span style="color:#94a3b8;font-size:.82em">No custom models registered yet.</span>')]
        return
    rows = []
    for key, info in _custom_models.items():
        col = info["color"]
        label = info["label"]
        swatch = f'<span style="display:inline-block;width:12px;height:12px;background:{col};border-radius:2px;margin-right:6px;vertical-align:middle"></span>'
        rm_btn = widgets.Button(description="Remove", button_style="danger", layout=widgets.Layout(width="74px", height="24px"))
        name_html = widgets.HTML(f'{swatch}<strong style="font-size:.85em">{label}</strong><code style="font-size:.76em;color:#64748b;margin-left:8px">key: {key}</code>')
        row = widgets.HBox([name_html, rm_btn], layout=widgets.Layout(align_items="center", margin="2px 0"))
        def _make_remover(k):
            def _remove(_):
                unregister_custom_model(k)
                if k in w_models:
                    del w_models[k]
                    _refresh_model_panel()
                _rebuild_registered_list()
            return _remove
        rm_btn.on_click(_make_remover(key))
        rows.append(row)
    _w_registered_list.children = rows




def _refresh_model_panel():
    rows = []
    for key, _name, _desc, _default in _MODEL_DEFS:
        if key in w_models:
            cb = w_models[key]
        else:
            continue
        col = _MODEL_COLORS.get(key, "#334155")
        rows.append(widgets.HBox([cb, widgets.HTML(f'<span style="color:#718096;font-size:.82em;margin-left:4px">— {_desc}</span>')],
                                 layout=widgets.Layout(align_items="center", margin="2px 0")))
    for key, info in _custom_models.items():
        if key not in w_models:
            col = info["color"]
            cb = widgets.Checkbox(value=True, description=info["label"], indent=False,
                                  layout=widgets.Layout(width="auto", margin="0"),
                                  style={"description_width":"0px","description_color":col})
            w_models[key] = cb
        else:
            cb = w_models[key]
        col = info["color"]
        rows.append(widgets.HBox([cb, widgets.HTML(f'<span style="color:#718096;font-size:.82em;margin-left:4px">— custom</span>')],
                                 layout=widgets.Layout(align_items="center", margin="2px 0")))
    w_model_box.children = rows




def _on_register(_):
    _w_register_status.value = ""
    key = _w_cm_key.value.strip()
    label = _w_cm_label.value.strip() or key
    formula = _w_cm_formula.value.strip()
    color = _w_cm_color.value
    if not key:
        _w_register_status.value = '<div class="qs-err">Model key is required.</div>'
        return
    if not formula:
        _w_register_status.value = '<div class="qs-err">Formula is required.</div>'
        return
    param_defs = []
    for rd in _cm_param_rows:
        name = rd["name"].value.strip()
        if name:
            param_defs.append((name, rd["p0"].value, rd["lo"].value, rd["hi"].value))
    try:
        register_custom_model(key, label, formula, param_defs, color=color)
        _w_register_status.value = f'<div class="qs-ok">Model "{label}" registered as <code>{key}</code>.</div>'
        _rebuild_registered_list()
        _refresh_model_panel()
        _w_cm_key.value = ""
        _w_cm_label.value = ""
        _w_cm_formula.value = ""
        _cm_param_rows.clear()
        _refresh_params_vbox()
    except Exception as exc:
        _w_register_status.value = f'<div class="qs-err"><strong>Registration failed:</strong> {exc}</div>'



_btn_register.on_click(_on_register)
_refresh_params_vbox()




def _lbl_cm(text):
    return widgets.HTML(f'<span class="qs-lbl">{text}</span>')




_cm_form = widgets.VBox([
    _lbl_cm("Model key  (short, no spaces)"), _w_cm_key,
    _lbl_cm("Display label"), _w_cm_label,
    _lbl_cm("Gamma(Q) formula  (meV) — uses q as the Q array"), _w_cm_formula,
    widgets.HTML('<div style="font-size:.78em;color:#64748b;margin:3px 0 8px 0">Available names: &nbsp;<code>q</code> &nbsp;<code>np</code> &nbsp;<code>hbar_mevps</code> &nbsp;<code>ce</code> &nbsp;<code>fickian</code> &nbsp;<code>ss_model</code> + your parameter names below</div>'),
    _lbl_cm("Colour"), _w_cm_color,
    _lbl_cm("Parameters"), _cm_params_vbox, _btn_add_param,
    widgets.HBox([_btn_register], layout=widgets.Layout(margin="10px 0 4px 0")),
    _w_register_status,
    _lbl_cm("Registered custom models"), _w_registered_list,
])




print("Custom model widgets ready.")

Custom model widgets ready.


In [ ]:
class FileBrowser:
    EXT = ".nxspe"

    def __init__(self, start_path="."):
        self._cwd = pathlib.Path(start_path).resolve()
        self._staged = {}

        self._w_path = widgets.HTML()
        self._btn_up = widgets.Button(description="Up", layout=widgets.Layout(width="56px", height="30px"))
        self._w_goto = widgets.Text(placeholder="Paste path and press Enter", continuous_update=False,
                                    layout=widgets.Layout(flex="1", height="30px"))
        self._btn_go = widgets.Button(description="Go", button_style="primary",
                                      layout=widgets.Layout(width="46px", height="30px"))
        self._nav = widgets.HBox([self._btn_up, self._w_goto, self._btn_go],
                                 layout=widgets.Layout(align_items="center", margin="0 0 6px 0"))

        self._w_dirs = widgets.Select(options=[], rows=4, layout=widgets.Layout(width="100%"))
        self._w_files = widgets.SelectMultiple(options=[], rows=6, layout=widgets.Layout(width="100%"))

        self._btn_add = widgets.Button(description="Add selected", button_style="success",
                                       layout=widgets.Layout(width="110px", height="28px"))
        self._btn_add_all = widgets.Button(description="Add all", layout=widgets.Layout(width="80px", height="28px"))
        self._btn_clr = widgets.Button(description="Clear all", button_style="warning",
                                       layout=widgets.Layout(width="90px", height="28px"))
        self._act = widgets.HBox([self._btn_add, self._btn_add_all, self._btn_clr],
                                 layout=widgets.Layout(margin="4px 0"))

        self._w_staged = widgets.SelectMultiple(options=[], rows=4, layout=widgets.Layout(width="100%"))
        self._btn_rm = widgets.Button(description="Remove selected", button_style="danger",
                                      layout=widgets.Layout(width="130px", height="26px"))
        self._rm_row = widgets.HBox([self._btn_rm], layout=widgets.Layout(margin="3px 0 0 0"))

        self._w_primary = widgets.Dropdown(options=[], description="", layout=widgets.Layout(width="100%"))
        self._w_status = widgets.HTML()





        def _lbl(text):
            return widgets.HTML(f'<span class="qs-lbl">{text}</span>')

        self.widget = widgets.VBox([
            self._w_path, self._nav,
            _lbl("Folders"), self._w_dirs,
            _lbl('Files  <span style="font-weight:400;color:#718096">(Ctrl-click / Shift-click for multi-select)</span>'),
            self._w_files, self._act,
            _lbl("Staged files"), self._w_staged, self._rm_row,
            _lbl('Primary file  <span style="font-weight:400;color:#718096">(INC reference file for spectrum plot)</span>'),
            self._w_primary, self._w_status,
        ])

        self._btn_up.on_click(self._go_up)
        self._btn_go.on_click(self._go_to)
        self._w_goto.observe(self._go_to, names="value")
        self._w_dirs.observe(self._on_dir_click, names="value")
        self._btn_add.on_click(self._add_highlighted)
        self._btn_add_all.on_click(self._add_all)
        self._btn_clr.on_click(lambda _: self._clear_all())
        self._btn_rm.on_click(self._remove_highlighted)
        self._w_primary.observe(self._on_primary, names="value")
        self._refresh()





    def _list(self, path):
        dirs, files = [], []
        try:
            entries = sorted(path.iterdir(), key=lambda e: (not e.is_dir(), e.name.lower()))
        except PermissionError:
            return [], []
        for e in entries:
            if e.name.startswith("."): continue
            if e.is_dir():
                try:
                    n = sum(1 for x in e.iterdir() if x.suffix == self.EXT)
                except PermissionError:
                    n = 0
                cnt = f"  [{n}]" if n else ""
                dirs.append((f"{e.name}/{cnt}", str(e)))
            elif e.suffix == self.EXT:
                kb = e.stat().st_size / 1024
                files.append((f"{e.name}  ({kb:.0f} KB)" if kb>0 else e.name, str(e)))
        return dirs, files




    def _refresh(self):
        dirs, files = self._list(self._cwd)
        self._w_path.value = f'<div class="qs-path">{self._cwd}</div>'
        self._w_dirs.options = dirs or [("(no sub-folders)", "")]
        self._w_files.options = files or [("(no .nxspe files here)", "")]
        self._refresh_staged()




    def _refresh_staged(self):
        staged = list(self._staged)
        names = [pathlib.Path(p).name for p in staged]
        self._w_staged.options = list(zip(names, staged)) if staged else [("(none staged)", "")]
        dd = [(pathlib.Path(p).name, p) for p in staged]
        prev = self._w_primary.value
        self._w_primary.options = dd if dd else [("—", "")]
        if prev and prev in staged:
            self._w_primary.value = prev
        elif staged:
            inc = [p for p in staged if "inc" in pathlib.Path(p).name.lower()]
            self._w_primary.value = inc[0] if inc else staged[0]
        n = len(staged)
        self._w_status.value = f'<span class="qs-badge">{n} file{"s" if n!=1 else ""} staged</span>' if n else '<span style="color:#94a3b8;font-size:.82em">No files staged</span>'




    def _go_up(self, _=None):
        p = self._cwd.parent
        if p != self._cwd:
            self._cwd = p
            self._refresh()




    def _go_to(self, _=None):
        raw = self._w_goto.value.strip()
        if not raw: return
        p = pathlib.Path(raw).expanduser().resolve()
        if p.is_dir():
            self._cwd = p
            self._w_goto.value = ""
            self._refresh()
        else:
            self._w_status.value = f'<span class="qs-warn">Path not found: {raw}</span>'



    def _on_dir_click(self, change):
        val = change.get("new", "")
        if val and pathlib.Path(val).is_dir():
            self._cwd = pathlib.Path(val)
            self._refresh()


    def _add_highlighted(self, _=None):
        for lbl, val in (self._w_files.options or []):
            if val in (self._w_files.value or ()) and val:
                self._staged[val] = True
        self._refresh_staged()

   
    def _add_all(self, _=None):
        for lbl, val in (self._w_files.options or []):
            if val and pathlib.Path(val).suffix == self.EXT:
                self._staged[val] = True
        self._refresh_staged()


    def _remove_highlighted(self, _=None):
        for val in (self._w_staged.value or ()):
            self._staged.pop(val, None)
        self._refresh_staged()


    def _clear_all(self):
        self._staged.clear()
        self._refresh_staged()


    def _on_primary(self, change):
        cb = getattr(self, '_preview_cb', None)
        if cb is not None and change.get('new'):
            cb()


    @property
    def selected_files(self):
        return list(self._staged)


    @property
    def primary(self):
        v = self._w_primary.value
        return v if (v and v not in ("", "—")) else (self.selected_files[0] if self.selected_files else None)


_browser = FileBrowser()

In [6]:
_MODEL_DEFS = [("ce",       "Chudley-Elliott",   "jump diffusion on a lattice",      True),
               ("fickian",  "Fickian",           "continuous Brownian motion",        True),
               ("ss_model", "Singwi-Sjolander",  "correlated jump diffusion",         False),
               ("lorentz",  "Lorentzian",        "phenomenological single Lorentzian",False),]


w_models = {}
_model_rows = []
for _key, _name, _desc, _default in _MODEL_DEFS:
    _col = _MODEL_COLORS[_key]
    cb = widgets.Checkbox(value=_default, description=_name, indent=False,
                          layout=widgets.Layout(width="auto", margin="0"),
                          style={"description_width":"0px", "description_color":_col})
    w_models[_key] = cb
    _model_rows.append(widgets.HBox([cb, widgets.HTML(f'<span style="color:#718096;font-size:.82em;margin-left:4px">— {_desc}</span>')],
                                    layout=widgets.Layout(align_items="center", margin="2px 0")))
w_model_box = widgets.VBox(_model_rows)
w_model_warn = widgets.HTML("")




def _chk_models(change):
    any_on = any(cb.value for cb in w_models.values())
    w_model_warn.value = '<div class="qs-warn" style="margin-top:6px">Select at least one model.</div>' if not any_on else ""
for _cb in w_models.values():
    _cb.observe(_chk_models, names="value")




w_method = widgets.ToggleButtons(
    options=[("Least Squares", "ls"), ("Bayesian MCMC", "bayes")],
    value="ls", style={"button_width":"136px", "font_weight":"600"})




_sl = dict(continuous_update=False, style={"description_width":"72px"}, layout=widgets.Layout(width="100%"))
w_nwalkers = widgets.IntSlider(value=32, min=8, max=128, step=8, description="Walkers", **_sl)
w_nwarmup  = widgets.IntSlider(value=300, min=50, max=2000, step=50, description="Burn-in", **_sl)
w_nkeep    = widgets.IntSlider(value=1000, min=100, max=5000, step=100, description="Samples", **_sl)
w_mcmc_box = widgets.VBox(
    [widgets.HTML('<span class="qs-lbl" style="margin-top:8px">MCMC settings</span>'),
     w_nwalkers, w_nwarmup, w_nkeep],
    layout=widgets.Layout(display="none"))




def _toggle_mcmc(change):
    w_mcmc_box.layout.display = "flex" if change["new"]=="bayes" else "none"
w_method.observe(_toggle_mcmc, names="value")




_rs = dict(continuous_update=False, readout_format=".2f", style={"description_width":"70px"}, layout=widgets.Layout(width="100%"))
w_qrange = widgets.FloatRangeSlider(value=[0.30,2.50], min=0.0, max=4.0, step=0.05, description="Q (1/A)", **_rs)
w_ewin = widgets.FloatSlider(value=0.80, min=0.20, max=2.0, step=0.05, description="+/-E (meV)", **_rs)
w_nbins = widgets.IntSlider(value=13, min=4, max=30, step=1, description="Q bins",
                            continuous_update=False, style={"description_width":"70px"}, layout=widgets.Layout(width="100%"))
w_q_target = widgets.BoundedFloatText(value=1.06, min=0.1, max=3.5, step=0.05, description="Q target",
                                      layout=widgets.Layout(width="180px"), style={"description_width":"68px"})



w_run = widgets.Button(description="Run Analysis", button_style="primary",
                       layout=widgets.Layout(width="180px", height="40px"))
w_prog = widgets.IntProgress(value=0, min=0, max=100, bar_style="info",
                             layout=widgets.Layout(width="100%", height="10px", display="none"))
w_prog_lbl = widgets.HTML("")



out_params = widgets.Output()
out_map    = widgets.Output()
out_fit    = widgets.Output()
out_hwhm   = widgets.Output()
out_post   = widgets.Output()
out_log    = widgets.Output()


print("Widgets created.")

Widgets created.


In [7]:
def _card(heading, *children):
    return widgets.VBox(
        [widgets.HTML(f'<div class="qs-heading">{heading}</div>')] + list(children),
        layout=widgets.Layout(border="1px solid #e2e8f0", border_radius="6px",
                              padding="12px 14px", margin="0 0 10px 0", background_color="#ffffff"))



def _lbl(txt, mt="0"):
    return widgets.HTML(f'<span class="qs-lbl" style="margin-top:{mt}">{txt}</span>')



panel_data = _card("Data Source", _browser.widget)
panel_models = _card("Physical Models", w_model_box, w_model_warn)
panel_custom = _card("Custom Model", _cm_form)
panel_fitting = _card("Fitting Method", w_method, w_mcmc_box)
panel_params = _card("Analysis Parameters",
    _lbl("Q range (\u00c5\u207b\u00b9)"), w_qrange,
    _lbl("Energy window", mt="6px"), w_ewin,
    _lbl("Q bins", mt="6px"), w_nbins,
    _lbl("Single-spectrum Q target (\u00c5\u207b\u00b9)", mt="6px"), w_q_target)



panel_run = widgets.VBox([
    w_run,
    widgets.HBox([w_prog, w_prog_lbl], layout=widgets.Layout(align_items="center", margin="8px 0 0 0"))
], layout=widgets.Layout(padding="14px 16px", border="1.5px solid #2563eb",
                         border_radius="6px", background_color="#eff6ff"))



left_col = widgets.VBox(
    [panel_data, panel_models, panel_custom, panel_fitting, panel_params, panel_run],
    layout=widgets.Layout(width="370px", min_width="350px", margin="0 14px 0 0"))



tabs = widgets.Tab(children=[out_params, out_map, out_fit, out_hwhm, out_post, out_log])
_tab_titles = ["Parameters", "S(Q,w) Map", "Fit / Residuals", "Gamma vs Q2", "Posteriors", "Run Log"]
for i, t in enumerate(_tab_titles):
    tabs.set_title(i, t)



right_col = widgets.VBox([tabs], layout=widgets.Layout(flex="1", min_width="0"))



_header = widgets.HTML(
    '<div style="border-left:4px solid #2563eb;padding:10px 16px;background:#f8fafc;margin-bottom:12px">'
    '<span style="font-size:1.15em;font-weight:700;color:#1e3a5f">QENS Analysis Studio</span>'
    '<span style="font-size:.85em;color:#64748b;margin-left:14px">Select files — choose models — run analysis</span>'
    '</div>')



app = widgets.VBox(
    [_header, widgets.HBox([left_col, right_col], layout=widgets.Layout(width="100%", align_items="flex-start"))],
    layout=widgets.Layout(width="100%"))



print("Layout assembled.")

Layout assembled.


In [8]:
def _prog(pct, msg=""):
    w_prog.value = int(pct)
    w_prog_lbl.value = f'<span style="font-size:.82em;color:#64748b;margin-left:6px">{msg}</span>'


def _log(msg, kind=""):
    cls = {"ok":"qs-log-ok","warn":"qs-log-warn","err":"qs-log-err"}.get(kind,"")
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    with out_log:
        display(HTML(f'<span class="qs-log-ts">[{ts}]</span><span class="{cls}">{msg}</span><br>'))


def _render(out_widget, fig, save_path=None):
    buf = _io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=140)
    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=200)
    plt.close(fig)
    buf.seek(0)
    img = widgets.Image(value=buf.read(), format="png", layout=widgets.Layout(width="100%", max_width="820px"))
    with out_widget:
        clear_output(wait=True)
        display(img)


def run_pipeline(_btn):
    for o in [out_params, out_map, out_fit, out_hwhm, out_post, out_log]:
        with o: clear_output(wait=True)
    w_run.disabled = True
    w_run.description = "Running..."
    w_prog.layout.display = "flex"
    w_prog.bar_style = "info"
    _prog(0, "Starting")

    sel_models = [k for k, cb in w_models.items() if cb.value]
    if not sel_models:
        with out_params:
            display(HTML('<div class="qs-warn">Select at least one model.</div>'))
        w_run.disabled = False
        w_run.description = "Run Analysis"
        return

    method = w_method.value
    q_lo, q_hi = w_qrange.value
    ewin = w_ewin.value
    nbins = w_nbins.value
    q_tgt = w_q_target.value

    run_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    rdir = os.path.join("output", run_ts)
    fdir = os.path.join(rdir, "figures")
    os.makedirs(fdir, exist_ok=True)
    _log(f"Output folder: {rdir}")
    _log(f"Models: {[_MODEL_LABELS.get(m,m) for m in sel_models]}")

    try:
        _prog(8, "Loading data")
        _log("Loading data...")
        staged = _browser.selected_files
        prim = _browser.primary
        if not staged:
            raise ValueError("No files staged.")
        if not prim:
            raise ValueError("No primary file set.")

        from collections import defaultdict
        by_dir = defaultdict(list)
        for p in staged:
            by_dir[str(pathlib.Path(p).parent)].append(pathlib.Path(p).name)

        dataset = {}
        for dpath, fnames in by_dir.items():
            pn = pathlib.Path(prim).name
            crits = [pn] if str(pathlib.Path(prim).parent)==dpath else []
            dataset.update(load_dataset(fnames, data_dir=dpath, critical_files=crits))
        if not dataset:
            raise RuntimeError("No files loaded.")

        for d in dataset.values():
            fit_elastic_peak(d)
        assign_resolution(dataset)

        pk = pathlib.Path(prim).name
        d_inc = dataset.get(pk, next(iter(dataset.values())))
        if pk not in dataset:
            _log(f"Primary not found; using {d_inc['name']}", "warn")
        _log(f"Loaded {len(dataset)} file(s).  Primary: {d_inc['name']}", "ok")

        _prog(18, "Configuring")
        cfg = Config(
            primary_file=d_inc["name"],
            q_min=q_lo, q_max=q_hi,
            n_bins=nbins, n_bins_mc=max(4, nbins-3),
            ewin_hwhm=ewin, ewin_mcmc=ewin,
            n_walkers=w_nwalkers.value,
            n_warmup=w_nwarmup.value,
            n_keep=w_nkeep.value,
            thin=5, random_seed=42, save_dir=rdir)
        cfg.to_json(os.path.join(rdir, "config.json"))

        _prog(26, "Rendering S(Q,w) map")
        _log("Rendering S(Q,w) map...")
        _render(out_map, fig_sqw_map(d_inc, ewin=min(ewin,1.0)),
                save_path=os.path.join(fdir, "sqw_map.pdf"))

        _prog(36, "Extracting HWHM")
        _log("Extracting HWHM...")
        q_hwhm, g_hwhm, g_err, eisf = extract_hwhm(d_inc, cfg)
        save_hwhm_csv(q_hwhm, g_hwhm, g_err, eisf, rdir)
        if len(q_hwhm) == 0:
            raise RuntimeError("No HWHM bins converged.")
        _log(f"  {len(q_hwhm)} bins  |  HWHM {g_hwhm.min()*1000:.0f}–{g_hwhm.max()*1000:.0f} µeV", "ok")

        n_m = len(sel_models)
        model_results = {}
        samples_map = {}
        best_model = None
        best_chi2r = np.inf
        best_d, best_l = 0.30, 2.50
        bayes_dbins = None

        for mi, model in enumerate(sel_models):
            p0 = 40 + mi * (48//n_m)
            p1 = 40 + (mi+1) * (48//n_m)
            mlbl = _MODEL_LABELS.get(model, model)
            _prog(p0, f"Fitting {mlbl}")
            _log(f"Fitting {mlbl}...")
            ls = _fit_gamma(q_hwhm, g_hwhm, model)
            model_results[model] = ls

            if "error" in ls:
                _log(f"  LS failed: {ls['error']}", "warn")
                continue

            D = ls.get("D", 0.30)
            L = ls.get("l", 2.50)
            _log(f"  D = {D:.5f}  ℓ = {L:.5f}  (LS)", "ok")

            if model in ("ce","fickian","ss_model"):
                try:
                    pred = (ce(q_hwhm, D, L) if model=="ce" else
                            fickian(q_hwhm, D) if model=="fickian" else
                            ss_model(q_hwhm, D, ls.get("tau_s",1.0)))
                    c2r = np.sum((g_hwhm-pred)**2 / np.where(pred>0, pred**2, 1e-30)) / max(len(q_hwhm)-2, 1)
                    ls["_chi2r"] = float(c2r)
                    if c2r < best_chi2r:
                        best_chi2r = c2r
                        best_model = model
                        best_d = D
                        best_l = L
                except Exception:
                    pass

            if method == "bayes" and model == "ce":
                _prog(p0 + (p1-p0)//2, "Bayesian MCMC for CE")
                _log("  Running Bayesian MCMC...")
                try:
                    if bayes_dbins is None:
                        bayes_dbins = build_data_bins(d_inc, cfg)
                    d_b, l_b, _ = find_map(bayes_dbins, d_inc["sigma_res"], cfg)
                    smp = run_mcmc(bayes_dbins, d_inc["sigma_res"], d_b, l_b, cfg)
                    samples_map[model] = smp
                    _log(f"  {len(smp)} posterior samples", "ok")
                    if model == best_model:
                        best_d, best_l = d_b, l_b
                except Exception as exc:
                    _log(f"  Bayesian failed: {exc}", "warn")

            _prog(p1, f"{mlbl} done")

        if best_model is None:
            best_model = sel_models[0]
            best_d = model_results[best_model].get("D", 0.30)
            best_l = model_results[best_model].get("l", 2.50)

        _prog(90, "Rendering fit")
        _log(f"Spectrum fit using {_MODEL_LABELS.get(best_model,best_model)}...")
        f_fit, chi2r = fig_fit_residuals(d_inc, best_d, best_l, q_target=q_tgt)
        _render(out_fit, f_fit, save_path=os.path.join(fdir, "fit_residuals.pdf"))

        _prog(93, "Rendering Gamma(Q)")
        _render(out_hwhm,
                fig_hwhm_comparison(q_hwhm, g_hwhm, g_err, model_results,
                                    samples_map=samples_map,
                                    res_hwhm_uev=d_inc["fwhm_res"]/2*1000),
                save_path=os.path.join(fdir, "hwhm_vs_q2.pdf"))

        if samples_map:
            _prog(96, "Rendering posteriors")
            fp = fig_posteriors_multi(samples_map, model_results, d_inc)
            if fp:
                _render(out_post, fp, save_path=os.path.join(fdir, "posteriors.pdf"))
        else:
            with out_post:
                display(HTML('<div class="qs-info" style="margin:16px">Posteriors only in Bayesian mode with CE.</div>'))

        _prog(98, "Compiling results")
        _build_params_table(d_inc, model_results, samples_map, method, chi2r, q_hwhm, rdir)
        _save_json(d_inc, model_results, samples_map, method, chi2r, rdir)

        _prog(100, "Complete")
        _log(f"Done. Results saved to: {rdir}", "ok")
        tabs.selected_index = 0

    except Exception as exc:
        with out_params:
            clear_output(wait=True)
            display(HTML(f'<div class="qs-err"><strong>Error:</strong> {exc}<br><pre>{traceback.format_exc()}</pre></div>'))
        _log(f"Error: {exc}", "err")
        w_prog.bar_style = "danger"
        _prog(w_prog.value, "Failed — see Parameters tab")
    finally:
        w_run.disabled = False
        w_run.description = "Run Analysis"






def _build_params_table(d_inc, model_results, samples_map, method, chi2r, q_hwhm, rdir):
    rows = [
        '<tr class="qs-section"><td colspan="4"><strong>Dataset</strong></td></tr>',
        f'<tr><td><strong>File</strong></td><td>{d_inc["name"]}</td><td></td><td></td></tr>',
        f'<tr><td><strong>Temperature</strong></td><td>{d_inc["temp"]} K</td><td></td><td></td></tr>',
        f'<tr><td><strong>Ei</strong></td><td>{d_inc["ei"]:.2f}</td><td>meV</td><td></td></tr>',
        f'<tr><td><strong>Resolution</strong></td><td>{d_inc["fwhm_res"]*1000:.1f}</td><td>µeV FWHM</td><td>{d_inc["res_source"]}</td></tr>',
        f'<tr><td><strong>Q range</strong></td><td>{q_hwhm.min():.3f} – {q_hwhm.max():.3f}</td><td>Å⁻¹</td><td></td></tr>',
        f'<tr><td><strong>Q bins</strong></td><td>{len(q_hwhm)}</td><td></td><td></td></tr>',
        '<tr class="qs-section"><td colspan="4"><strong>Fitting</strong></td></tr>',
        f'<tr><td><strong>Method</strong></td><td>{"Bayesian MCMC" if method=="bayes" else "Least Squares"}</td><td></td><td></td></tr>',
        f'<tr><td><strong>Spectrum χ²ᵣ</strong></td><td>{chi2r:.3f}</td><td></td><td>ideal ≈ 1</td></tr>',
    ]
    for model, ls in model_results.items():
        rows.append(f'<tr class="qs-model-{model}"><td colspan="4"><strong>{_MODEL_LABELS.get(model,model)}</strong></td></tr>')
        if "error" in ls:
            rows.append(f'<tr><td colspan="4" style="color:#b91c1c">Fit failed: {ls["error"]}</td></tr>')
            continue
        smp = samples_map.get(model)
        if method=="bayes" and smp is not None:
            ds = smp[:,0]; ls_ = np.abs(smp[:,1]); ts = ls_**2/(6*ds)
            for arr, nm, un in [(ds,"D","Å² ps⁻¹"),(ls_,"ℓ","Å"),(ts,"τ","ps")]:
                med = np.median(arr)
                lo, hi = np.percentile(arr, [2.5,97.5])
                rows.append(f'<tr><td><strong>{nm}</strong></td><td>{med:.5f}</td><td>{un}</td><td>95% CI [{lo:.5f}, {hi:.5f}]</td></tr>')
        else:
            D = ls.get("D")
            if D is not None:
                rows.append(f'<tr><td><strong>D</strong></td><td>{D:.5f}</td><td>Å² ps⁻¹</td><td></td></tr>')
            L = ls.get("l")
            if L is not None:
                rows.append(f'<tr><td><strong>ℓ</strong></td><td>{L:.5f}</td><td>Å</td><td></td></tr>')
                tau = ls.get("tau", L**2/(6*D) if D else 0)
                rows.append(f'<tr><td><strong>τ</strong></td><td>{tau:.5f}</td><td>ps</td><td></td></tr>')
            ts = ls.get("tau_s")
            if ts is not None:
                rows.append(f'<tr><td><strong>τₛ</strong></td><td>{ts:.5f}</td><td>ps</td><td></td></tr>')
            mh = ls.get("mean_hwhm_mev")
            if mh is not None:
                rows.append(f'<tr><td><strong>Mean HWHM</strong></td><td>{mh*1000:.1f}</td><td>µeV</td><td></td></tr>')
            if model in _custom_models:
                info = _custom_models[model]
                for pd in info["param_defs"]:
                    val = ls.get(pd[0])
                    if val is not None:
                        rows.append(f'<tr><td><strong>{pd[0]}</strong></td><td>{val:.5f}</td><td></td><td></td></tr>')
        c2 = ls.get("_chi2r")
        if c2 is not None:
            rows.append(f'<tr><td><strong>χ²ᵣ</strong></td><td>{c2:.4f}</td><td></td><td>ideal ≈ 1</td></tr>')
    table = f'<table class="qs-tbl"><thead><tr><th>Parameter</th><th>Value</th><th>Unit</th><th>Notes</th></tr></thead><tbody>{"".join(rows)}</tbody></table>'
    banner = f'<div class="qs-ok" style="margin-bottom:12px">Analysis complete — results saved to <code>{rdir}</code></div>'
    with out_params:
        clear_output(wait=True)
        display(HTML(banner + table))

def _save_json(d_inc, model_results, samples_map, method, chi2r, rdir):
    out = {
        "timestamp": datetime.datetime.now().isoformat(),
        "dataset": d_inc["name"],
        "temperature_K": int(d_inc["temp"]),
        "Ei_meV": float(d_inc["ei"]),
        "method": method,
        "res_fwhm_meV": float(d_inc["fwhm_res"]),
        "res_source": d_inc["res_source"],
        "spectrum_chi2r": float(chi2r),
        "models": {}
    }
    for model, ls in model_results.items():
        entry = {"ls": {k: float(v) for k,v in ls.items() if k not in ("cov","error","_chi2r")}}
        if "_chi2r" in ls:
            entry["gamma_chi2r"] = float(ls["_chi2r"])
        smp = samples_map.get(model)
        if smp is not None:
            ds = smp[:,0]; ls_ = np.abs(smp[:,1]); ts = ls_**2/(6*ds)
            def _ci(a):
                lo, hi = np.percentile(a, [2.5,97.5])
                return {"median": float(np.median(a)), "lo95": float(lo), "hi95": float(hi)}
            entry["bayesian"] = {"n_samples": int(len(ds)), "D": _ci(ds), "l": _ci(ls_), "tau": _ci(ts)}
            np.save(os.path.join(rdir, f"posterior_{model}.npy"), smp)
        out["models"][model] = entry
    with open(os.path.join(rdir, "results.json"), "w") as fh:
        json.dump(out, fh, indent=2)

def _preview_primary():
    prim = _browser.primary
    if not prim or not os.path.exists(prim):
        return
    with out_map:
        clear_output(wait=True)
        display(HTML('<div class="qs-info" style="margin:16px 0">Loading preview…</div>'))
    try:
        d = read_nxspe_hdf5(prim)
        fit_elastic_peak(d)
        assign_resolution({d["name"]: d})
        fig = fig_data_preview(d)
        _render(out_map, fig)
        tabs.selected_index = 1
        _log(f"Preview: {d['name']}  Ei={d['ei']:.2f} meV  T={d['temp']} K  good={len(d['good'])} det", "ok")
    except Exception as exc:
        with out_map:
            clear_output(wait=True)
            display(HTML(f'<div class="qs-err"><strong>Preview failed:</strong> {exc}<br><span style="font-size:.82em">Call <code>inspect_nxspe(path)</code> to check file structure.</span></div>'))
        _log(f"Preview error: {exc}", "err")

_browser._preview_cb = _preview_primary
w_run.on_click(run_pipeline)
print("Pipeline wired. Ready.")

Pipeline wired. Ready.


In [9]:
display(HTML(_CSS))
display(app)
display(HTML('<div class="qs-info" style="margin-top:10px;max-width:940px;font-size:.85em">'
             '<strong>File browser:</strong> &nbsp;Click a folder to navigate into it &middot; '
             '<em>Up</em> to go back &middot; Ctrl-click / Shift-click in the file list for multi-select &middot; '
             '<em>Add selected</em> stages highlighted files &middot; <em>Add all</em> stages every .nxspe in the current folder &middot; '
             'Set the <em>Primary file</em> dropdown to your INC reference.'
             '</div>'))